# 08 - UCI AI4I 2020 Machine Failure Classification
### Phase 2: Machine Learning Baseline Development & Evaluation

This notebook evaluates machine failure classification on the tabular **UCI AI4I 2020** dataset:
- Severe class imbalance: ~3.39% failure rate.
- Input features strictly exclude all target leakage columns (`UDI`, `Product ID`, `Machine failure`, `TWF`, `HDF`, `PWF`, `OSF`, `RNF`).
- Domain-informed physical features are incorporated.
- Decision threshold is tuned strictly on the validation subset.
- Secondary failure-mode breakdown (TWF, HDF, PWF, OSF, RNF) is performed for post-hoc diagnostic explanation.


In [ ]:
import sys
from pathlib import Path
import joblib
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

project_root = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.append(str(project_root / "src"))

from evaluation.metrics import compute_classification_metrics, find_optimal_threshold
from training.ai4i_trainer import load_ai4i_processed_data


## 1. Load Processed AI4I Data


In [ ]:
proc_dir = project_root / "data" / "processed" / "ai4i"
X_train_full, y_train_full, X_test, y_test, df_eng = load_ai4i_processed_data(proc_dir)

print(f"X_train Shape: {X_train_full.shape}, Failure Rate: {y_train_full.mean()*100:.2f}%")
print(f"X_test Shape:  {X_test.shape}, Failure Rate:  {y_test.mean()*100:.2f}%")
print(f"Features: {list(X_train_full.columns)}")


## 2. Load Models & Evaluate on Test Set


In [ ]:
models_dir = project_root / "models" / "baseline"
models = ["Logistic Regression", "Random Forest Classifier", "Gradient Boosting Classifier", "XGBoost Classifier"]

summary = []
loaded_clfs = {}

for m in models:
    clean_m = m.lower().replace(" ", "_")
    p = models_dir / f"ai4i_clf_{clean_m}.joblib"
    meta_p = models_dir / f"ai4i_clf_{clean_m}_meta.json"
    if p.exists() and meta_p.exists():
        with open(meta_p, "r") as f:
            meta = json.load(f)
        clf = joblib.load(p)
        loaded_clfs[m] = (clf, meta["optimal_threshold"])
        summary.append({
            "Model": m,
            "Tuned Threshold": meta["optimal_threshold"],
            "Val PR-AUC": meta["val_metrics"]["val_pr_auc"],
            "Val F1": meta["val_metrics"]["val_f1"],
            "Test PR-AUC": meta["test_metrics"]["test_pr_auc"],
            "Test F1": meta["test_metrics"]["test_f1"],
            "Test Recall": meta["test_metrics"]["test_recall"],
            "Test Precision": meta["test_metrics"]["test_precision"],
            "Test ROC-AUC": meta["test_metrics"]["test_roc_auc"],
        })

df_ai4i_results = pd.DataFrame(summary)
display(df_ai4i_results)


## 3. Secondary Failure-Mode Analysis


In [ ]:
# Check detection rates across specific failure modes
best_m = df_ai4i_results.sort_values(by="Val PR-AUC", ascending=False).iloc[0]["Model"]
clf, th = loaded_clfs[best_m]

X_test_np = X_test.to_numpy(dtype=np.float32)
y_prob = clf.predict_proba(X_test_np)[:, 1] if hasattr(clf, "predict_proba") else 1.0 / (1.0 + np.exp(-clf.decision_function(X_test_np)))
y_pred = (y_prob >= th).astype(int)

from sklearn.model_selection import train_test_split as tts
_, test_idx = tts(np.arange(len(df_eng)), test_size=0.20, random_state=42, stratify=df_eng["Machine failure"])
df_test_modes = df_eng.iloc[test_idx].reset_index(drop=True)

modes = ["TWF", "HDF", "PWF", "OSF", "RNF"]
mode_breakdown = []
for m in modes:
    mask = df_test_modes[m] == 1
    total = int(mask.sum())
    detected = int(y_pred[mask].sum()) if total > 0 else 0
    rec = detected / total if total > 0 else 0.0
    mode_breakdown.append({"Failure Mode": m, "Total in Test": total, "Detected": detected, "Recall (%)": round(rec*100, 1)})

df_mode = pd.DataFrame(mode_breakdown)
display(df_mode)
